# Data cleaning pipeline

This notebook builds MatchCast's processed match table from the raw
`martj42/international_results` snapshot. It is built up section by section,
one cleaning concern at a time, matching the schema defined in
`01_processed_match_schema.ipynb`. Each section documents its rule and
validates it before the next section runs.

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/international_results.csv")
PROCESSED_PATH = Path("../data/processed/matches.csv")

raw = pd.read_csv(RAW_PATH, dtype="string", keep_default_na=False)
raw["source_row_number"] = range(len(raw))

print(f"Loaded {len(raw)} raw rows from {RAW_PATH}")

Loaded 49501 raw rows from ..\data\raw\international_results.csv


## Parse and sort match dates

Dates are parsed with an explicit `YYYY-MM-DD` format so malformed values
surface as `NaT` instead of being silently misparsed. Rows are then sorted by
`date` using a stable sort with `source_row_number` as the deterministic
tiebreaker, so matches played on the same day keep their original upstream
order every time this notebook runs.

In [2]:
matches = raw.copy()
matches["date"] = pd.to_datetime(
    matches["date"].str.strip(), format="%Y-%m-%d", errors="raise"
)
matches = matches.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)

print(f"Date range: {matches['date'].min().date()} to {matches['date'].max().date()}")

Date range: 1872-11-30 to 2026-07-06


### Validation: date parsing and sort determinism

In [3]:
assert matches["date"].notna().all(), "every row must have a parsed date"
assert len(matches) == len(raw), "sorting must not drop or add rows"
assert matches["date"].is_monotonic_increasing, "rows must be sorted by date"

resorted = matches.sort_values(
    by=["date", "source_row_number"], kind="stable"
).reset_index(drop=True)
assert matches["source_row_number"].tolist() == resorted["source_row_number"].tolist(), (
    "re-sorting the already-sorted table must be a no-op (deterministic order)"
)

same_day = matches[matches.duplicated(subset="date", keep=False)]
for _, group in same_day.groupby("date"):
    assert group["source_row_number"].is_monotonic_increasing, (
        "same-day matches must preserve original upstream order"
    )

print("Date parsing and deterministic sort validated.")

Date parsing and deterministic sort validated.
